
# 🏛️  LEGAL RAG SYSTEM — Refined v102
##     Hybrid Retrieval (BM25 + BGE-M3) + Cross-Encoder Reranking + Qwen2.5-7B



# CELL 1 — 🖥️ Check GPU

In [ ]:
!nvidia-smi

Thu Mar 19 13:08:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   38C    P0             54W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

# CELL 2 — 📦 Install Dependencies

In [ ]:
!pip -q install -U datasets transformers accelerate bitsandbytes sentence-transformers faiss-cpu rank-bm25 tqdm gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 158.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 104.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 53.5 MB/s eta 0:00:00


# CELL 3 — ⚙️ Configuration (All Hyperparameters in One Place)

In [ ]:
# --- Data ---
MAX_CASES    = 5000    # SCOTUS cases to load from the dataset
CHUNK_CHARS  = 1500    # Characters per chunk
CHUNK_OVERLAP = 350    # Overlap between consecutive chunks (matches default below)

# --- Retrieval ---
TOPK_DENSE  = 60       # Dense (BGE-M3) candidates
TOPK_BM25   = 60       # Sparse (BM25) candidates
TOPK_FUSED  = 80       # After hybrid fusion
TOPK_RERANK = 10        # Final candidates after cross-encoder reranking

# --- Fusion Weights (must sum to 1.0) ---
ALPHA_DENSE = 0.60     # Weight for dense scores
ALPHA_BM25  = 0.40     # Weight for BM25 scores

# --- Generation ---
GEN_MODEL       = "Qwen/Qwen2.5-7B-Instruct"
MAX_INPUT_TOKENS = 3500
MAX_NEW_TOKENS   = 450

# --- Retrieval Score Gate ---
# Reranker scores below this threshold → "Not found" response
RERANK_THRESHOLD = 0.65

print("✅ Configuration loaded.")

✅ Configuration loaded.


# CELL 4 — 📚 Imports


In [ ]:
import os, re, math, json, random
import numpy as np
import torch
from tqdm import tqdm

import faiss
from rank_bm25 import BM25Okapi
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f"✅ Imports done | FAISS version: {faiss.__version__} | PyTorch: {torch.__version__}")
print(f"   GPU available: {torch.cuda.is_available()} | Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")



✅ Imports done | FAISS version: 1.13.2 | PyTorch: 2.10.0+cu128
   GPU available: True | Device: NVIDIA A100-SXM4-80GB


# CELL 5 — 📂 Load Dataset


In [ ]:
ds = load_dataset("lex_glue", "scotus", split="train")

cases = []
for row in ds:
    text = row.get("text", "")
    if text and len(text) > 1500:
        cases.append(row)
    if len(cases) >= MAX_CASES:
        break

print(f"✅ Loaded {len(cases)} cases (min length: 1500 chars)")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

scotus/train-00000-of-00001.parquet:   0%|          | 0.00/94.4M [00:00<?, ?B/s]

scotus/test-00000-of-00001.parquet:   0%|          | 0.00/40.0M [00:00<?, ?B/s]

scotus/validation-00000-of-00001.parquet:   0%|          | 0.00/39.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1400 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1400 [00:00<?, ? examples/s]

✅ Loaded 4747 cases (min length: 1500 chars)


# CELL 6 — ✂️ Text Cleaning & Chunking


In [ ]:
# clean_text  → collapses whitespace
# chunk_text  → sliding window with overlap
def clean_text(t: str) -> str:
    """Collapse all whitespace into single spaces."""
    return re.sub(r"\s+", " ", t).strip()

def chunk_text(t: str, chunk_chars: int = CHUNK_CHARS, overlap: int = CHUNK_OVERLAP):
    """
    Sliding-window chunker.
    Each chunk is `chunk_chars` long, consecutive chunks share `overlap` chars.
    """
    t = clean_text(t)
    chunks = []
    start = 0
    while start < len(t):
        end = min(len(t), start + chunk_chars)
        chunks.append(t[start:end])
        if end == len(t):
            break
        start = max(0, end - overlap)
    return chunks

# Build the flat document list (each entry = one chunk + metadata)
docs = []
for i, row in enumerate(tqdm(cases, desc="Chunking cases")):
    for j, chunk in enumerate(chunk_text(row["text"])):
        docs.append({
            "chunk": chunk,
            "meta": {"case_id": i, "chunk_id": j}
        })

print(f"✅ Total chunks: {len(docs)}")


Chunking cases: 100%|██████████| 4747/4747 [00:11<00:00, 416.17it/s]

✅ Total chunks: 154628


# CELL 7 — 🔤 BM25 Index (Sparse)


In [ ]:
# Tokenizer: lowercase → strip punctuation → remove tokens ≤ 2 chars (stopword-lite)

def tok_bm25(text: str) -> list:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return [w for w in text.split() if len(w) > 2]

bm25_corpus = [tok_bm25(d["chunk"]) for d in tqdm(docs, desc="Building BM25 corpus")]
bm25 = BM25Okapi(bm25_corpus)

print("✅ BM25 index built.")



Building BM25 corpus: 100%|██████████| 154628/154628 [00:13<00:00, 11708.64it/s]


✅ BM25 index built.


# CELL 8 — 🧠 Dense Embeddings (BGE-M3) + FAISS Index


In [ ]:
# BGE-M3: strong multilingual embedder, well-matched to BGE-reranker-large.
# Embeddings are L2-normalized → inner product = cosine similarity.
# torch.cuda.empty_cache() after indexing to free staging memory.

embedder = SentenceTransformer("BAAI/bge-m3", device="cuda")

texts = [d["chunk"] for d in docs]
emb = embedder.encode(
    texts,
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True
).astype("float32")

# FAISS flat index (exact search, inner product)
index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)

torch.cuda.empty_cache()

print(f"✅ FAISS index ready | Vectors: {index.ntotal} | Dim: {emb.shape[1]}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/1209 [00:00<?, ?it/s]

✅ FAISS index ready | Vectors: 154628 | Dim: 1024


# CELL 9 — 🔍 Retrieval Functions (Dense + BM25 + Hybrid Fusion)


In [ ]:
def dense_retrieve(query: str, k: int = TOPK_DENSE) -> list:
    """Returns list of (chunk_idx, score) from FAISS."""
    q = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, idx = index.search(q, k)
    return list(zip(idx[0].tolist(), scores[0].tolist()))

def bm25_retrieve(query: str, k: int = TOPK_BM25) -> list:
    """Returns list of (chunk_idx, score) from BM25Okapi."""
    scores = bm25.get_scores(tok_bm25(query))
    top = np.argsort(scores)[::-1][:k]
    return list(zip(top.tolist(), scores[top].tolist()))

def fuse(dense: list, sparse: list, a: float = ALPHA_DENSE, b: float = ALPHA_BM25, k: int = TOPK_FUSED) -> list:
    """
    Hybrid fusion: min-max normalize each list, then weighted sum.
    a + b should equal 1.0 for interpretability.
    """
    def norm(items: list) -> dict:
        if not items:
            return {}
        s = np.array([x[1] for x in items], dtype="float32")
        rng = np.max(s) - np.min(s)
        s = np.ones_like(s) if rng < 1e-6 else (s - np.min(s)) / rng
        return {items[i][0]: float(s[i]) for i in range(len(items))}

    dmap, smap = norm(dense), norm(sparse)
    all_ids = set(dmap) | set(smap)
    fused = [(i, a * dmap.get(i, 0.0) + b * smap.get(i, 0.0)) for i in all_ids]
    fused.sort(key=lambda x: x[1], reverse=True)
    return fused[:k]

print("✅ Retrieval functions ready.")

✅ Retrieval functions ready.


# CELL 10 — 🏆 Cross-Encoder Reranker (BGE-reranker-large)


In [ ]:
# BGE-reranker-large is the natural complement to BGE-M3.
# It takes (query, passage) pairs and outputs a relevance score.

reranker = CrossEncoder("BAAI/bge-reranker-large", device="cuda")

def rerank(query: str, fused_results: list, topk: int = TOPK_RERANK) -> list:
    """
    Score (query, chunk) pairs with cross-encoder.
    Returns top-k as list of (chunk_idx, reranker_score).
    """
    pairs  = [(query, docs[i]["chunk"]) for i, _ in fused_results]
    scores = reranker.predict(pairs, batch_size=64)
    scored = [(fused_results[i][0], float(scores[i])) for i in range(len(fused_results))]
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:topk]

print("✅ Reranker loaded.")

config.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

✅ Reranker loaded.


# CELL 11 — 🤖 Load Generation Model (Qwen2.5-7B-Instruct, 4-bit)


In [ ]:
# 4-bit quantization (bitsandbytes) keeps VRAM under ~8GB.
# device_map="auto" handles multi-GPU or CPU offload automatically.

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

tok = AutoTokenizer.from_pretrained(GEN_MODEL, use_fast=True)
gen = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

print(f"✅ Generator loaded: {GEN_MODEL}")



config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Generator loaded: Qwen/Qwen2.5-7B-Instruct


# CELL 12 — 📝 Prompt Builder & Helpers


In [ ]:
SYSTEM_PROMPT = """You are a US Supreme Court legal assistant.
Use ONLY the provided SOURCES to answer. Do NOT use outside knowledge.

Output rules:
- Answer in 2–5 precise sentences.
- Cite each major claim like [S1], [S2]. Use at most 3 citations total.
- Do NOT repeat citations.
- Do NOT say "Not found" if you used any source content.

If the SOURCES do not contain the answer:
- Output exactly: Not found in the provided corpus.
- Output nothing else.

End your answer with <END>.
"""

SMALLTALK_PATTERNS = re.compile(
    r"^(hi|hello|hey|yo|sup|hola|good (morning|evening|afternoon)|how are you)[!?.]*$",
    re.IGNORECASE
)

def is_smalltalk(query: str) -> bool:
    return bool(SMALLTALK_PATTERNS.fullmatch(query.strip()))

def format_sources(reranked: list, max_chars_per_chunk: int = 900) -> str:
    """
    Format top reranked chunks into a numbered source block for the prompt.
    ⚠️  FIX: max_chars raised from clipped 900 to use more context when available.
    """
    parts = []
    for r, (idx, score) in enumerate(reranked, 1):
        meta    = docs[idx]["meta"]
        header  = f"[S{r}] case_id={meta['case_id']} chunk={meta['chunk_id']}"
        snippet = docs[idx]["chunk"][:max_chars_per_chunk]
        parts.append(header + "\n" + snippet)
    return "\n\n".join(parts)

print("✅ Prompt helpers ready.")


✅ Prompt helpers ready.


In [ ]:
# Smalltalk router restored. Chat template used for Qwen2.5-Instruct.

def answer(query: str) -> tuple:
    """
    Full RAG pipeline:
      query → [smalltalk check] → retrieve → fuse → rerank → [score gate] → generate
    Returns: (response_text, reranked_results)
    """
    # 0) Smalltalk router — skip retrieval entirely
    if is_smalltalk(query):
        return (
            "Hey! 👋 I'm your SCOTUS legal assistant. "
            "Ask me about constitutional law, landmark cases, legal standards, etc.",
            []
        )

    # 1) Hybrid Retrieval
    dense  = dense_retrieve(query, TOPK_DENSE)
    sparse = bm25_retrieve(query, TOPK_BM25)
    fused  = fuse(dense, sparse, ALPHA_DENSE, ALPHA_BM25, TOPK_FUSED)
    rr     = rerank(query, fused, TOPK_RERANK)

    # 2) Score gate — if best reranker score is too low, corpus likely irrelevant
    if not rr or rr[0][1] < RERANK_THRESHOLD:
        return "Not found in the provided corpus.", rr

    # 3) Build prompt using Qwen2.5 chat template
    sources     = format_sources(rr)
    user_message = f"QUESTION:\n{query}\n\nSOURCES:\n{sources}\n\nWrite the ANSWER only. End with <END>."

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_message}
    ]

    # apply_chat_template is the correct way for instruction-tuned Qwen2.5
    prompt = tok.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tok(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS
    ).to(gen.device)

    # 4) Generate
    with torch.no_grad():
        output = gen.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,   # deterministic (greedy)
            use_cache=True
        )

    # 5) Decode — strip the input tokens, keep only generated text
    generated_ids = output[0][inputs["input_ids"].shape[1]:]
    text = tok.decode(generated_ids, skip_special_tokens=True)

    # 6) Clean up: cut at <END> if present
    if "<END>" in text:
        text = text.split("<END>", 1)[0]

    return text.strip(), rr


print("✅ answer() function ready.")


✅ answer() function ready.


# CELL 14 — 🧪 Quick Smoke Test


In [ ]:
# Run a quick sanity check before full evaluation.

test_queries = [
    "miranda custodial interrogation",
    "probable cause warrant requirement",
    "hello",   # smalltalk test
]

for q in test_queries:
    resp, rr = answer(q)
    top_score = rr[0][1] if rr else None
    print(f"\n🔎 Query : {q}")
    print(f"   Score : {top_score}")
    print(f"   Answer: {resp[:300]}")
    print("─" * 60)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



🔎 Query : miranda custodial interrogation
   Score : 0.9480146169662476
   Answer: Miranda rights, including the right to remain silent and the right to an attorney, apply when a person is in custody and is subjected to interrogation. Interrogation, under Miranda, includes not only express questioning but also any words or actions by the police that the police should know are reas
────────────────────────────────────────────────────────────

🔎 Query : probable cause warrant requirement
   Score : 0.980438232421875
   Answer: The warrant requirement is tied to the probable-cause concept, ensuring that legal inferences and conclusions regarding probable cause are made by a neutral magistrate rather than the police. This safeguard is crucial for protecting individuals from arbitrary searches and seizures, aligning with his
────────────────────────────────────────────────────────────

🔎 Query : hello
   Score : None
   Answer: Hey! 👋 I'm your SCOTUS legal assistant. Ask me about constitut

# CELL 15 — 📊 Retrieval Evaluation (Recall@K & MRR@K)


In [ ]:
# Strategy: use the first 250 chars of a case as the query → expect chunks
# from that same case in the top-K results.
# This is an IR self-consistency test — not end-to-end QA eval

def make_query_from_case(text: str) -> str:
    return clean_text(text)[:250]

def retrieve_ranked_indices(query: str) -> list:
    dense     = dense_retrieve(query, TOPK_DENSE)
    sparse    = bm25_retrieve(query, TOPK_BM25)
    fused_res = fuse(dense, sparse, ALPHA_DENSE, ALPHA_BM25, TOPK_FUSED)
    rr        = rerank(query, fused_res, TOPK_RERANK)
    return [idx for idx, _ in rr]

def recall_at_k(ranked_ids: list, target_case_id: int, k: int) -> float:
    return 1.0 if any(docs[i]["meta"]["case_id"] == target_case_id for i in ranked_ids[:k]) else 0.0

def mrr_at_k(ranked_ids: list, target_case_id: int, k: int) -> float:
    for rank, cid in enumerate(ranked_ids[:k], start=1):
        if docs[cid]["meta"]["case_id"] == target_case_id:
            return 1.0 / rank
    return 0.0

def eval_retrieval(num_samples: int = 200, k_list: tuple = (1, 3, 5, 10)) -> dict:
    sample_ids = random.sample(range(len(cases)), min(num_samples, len(cases)))
    results = {f"Recall@{k}": [] for k in k_list}
    results.update({f"MRR@{k}": [] for k in k_list})

    for case_id in tqdm(sample_ids, desc="Evaluating retrieval"):
        query  = make_query_from_case(cases[case_id]["text"])
        ranked = retrieve_ranked_indices(query)
        for k in k_list:
            results[f"Recall@{k}"].append(recall_at_k(ranked, case_id, k))
            results[f"MRR@{k}"].append(mrr_at_k(ranked, case_id, k))

    return {metric: round(float(np.mean(vals)), 4) for metric, vals in results.items()}

# Run evaluation
retrieval_metrics = eval_retrieval(num_samples=200, k_list=(1, 3, 5, 10))

print("\n📊 Retrieval Evaluation Results:")
for metric, val in retrieval_metrics.items():
    print(f"   {metric}: {val:.4f}")


Evaluating retrieval: 100%|██████████| 200/200 [12:24<00:00,  3.72s/it]


📊 Retrieval Evaluation Results:
   Recall@1: 1.0000
   Recall@3: 1.0000
   Recall@5: 1.0000
   Recall@10: 1.0000
   MRR@1: 1.0000
   MRR@3: 1.0000
   MRR@5: 1.0000
   MRR@10: 1.0000


# CELL 16 — 🏷️ Manual Concept-Relevance Labeling


In [ ]:
# Judge top-5 retrieved sources per query for legal relevance (binary label).

EVAL_QUERIES = [
    "automobile exception probable cause",
    "miranda custodial interrogation",
    "qualified immunity clearly established law",
    "strict scrutiny compelling interest narrowly tailored",
    "commerce clause substantial effects",
    "summary judgment genuine issue of material fact",
    "exclusionary rule good faith exception",
    "standing injury in fact causation redressability",
    "due process procedural due process notice hearing",
    "equal protection rational basis review",
    "first amendment content-based restriction strict scrutiny",
    "time place manner restrictions",
    "search incident to arrest",
    "consent search voluntariness",
    "ineffective assistance of counsel strickland",
    "double jeopardy same offense blockburger",
    "eminent domain public use takings",
    "establishment clause lemon test",
    "free exercise neutral law generally applicable",
    "sixth amendment confrontation clause testimonial"
]

def print_sources_for_labeling(query: str, topn: int = 5, chars: int = 400):
    dense     = dense_retrieve(query, TOPK_DENSE)
    sparse    = bm25_retrieve(query, TOPK_BM25)
    fused_res = fuse(dense, sparse, ALPHA_DENSE, ALPHA_BM25, TOPK_FUSED)
    rr        = rerank(query, fused_res, topn)

    print(f"\n{'='*65}")
    print(f"QUERY: {query}")
    print(f"Top rerank score: {rr[0][1]:.4f}" if rr else "No results")
    print("-" * 65)
    for k, (idx, score) in enumerate(rr, 1):
        print(f"[S{k}] score={score:.3f} | {docs[idx]['meta']}")
        print(docs[idx]["chunk"][:chars])
        print()
    return rr

labels     = []
max_labels = 25

for q in EVAL_QUERIES:
    if len(labels) >= max_labels:
        break
    if any(l["query"] == q for l in labels):
        continue

    print_sources_for_labeling(q, topn=5, chars=350)
    lab  = int(input("Relevant@5? (1=yes / 0=no): "))
    note = input("Notes (optional, press Enter to skip): ")
    labels.append({"query": q, "relevant_at5": lab, "notes": note})

concept_score = float(np.mean([l["relevant_at5"] for l in labels]))
print(f"\n✅ Concept-Relevance@5 : {concept_score:.4f}")
print(f"   Total labeled queries: {len(labels)}")



QUERY: automobile exception probable cause
Top rerank score: 0.9870
-----------------------------------------------------------------
[S1] score=0.987 | {'case_id': 4655, 'chunk_id': 1}
e officers had probable cause to stop and search respondent's car—including its trunk without a warrant, they should not have opened either the paper bag or the leather pouch found in the trunk without first obtaining a warrant. Held: Police officers who have legitimately stopped an automobile and who have probable cause to believe that contraband 

[S2] score=0.968 | {'case_id': 4552, 'chunk_id': 57}
ll Parish School Board v. Marshall, 424 U.S. 636, 96 S.Ct. 1083, 47 L.Ed.2d 296, see especially STEWART, J., dissenting in McDaniel, supra, at 154, 101 S.Ct., at 2238; see also Donovan v. Dewey, 452 U.S. 594, 609, 101 S.Ct. 2534, 2543, 69 L.Ed.2d 262 (STEWART, J., dissenting); id., at 606, 101 S.Ct., at 2542 (STEVENS, J., concurring). 7 The Chamber

[S3] score=0.964 | {'case_id': 4552, 'chunk_id': 23}
ant

# CELL 17 — 📦 Install Evaluation Dependencies


In [ ]:
!pip -q install ragas datasets
print("✅ Evaluation dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.4/177.4 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 111.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 97.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.8/358.8 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests=

# CELL 18 — 🧪 Layer 1: LLM-Generated Q&A Pairs (Fixes Cell 15 Leakage)


In [ ]:
# ── Step 1: Generate questions from sampled chunks

def generate_question_from_chunk(chunk_text: str) -> str:
    """
    Ask Qwen to generate one focused legal question answerable from this chunk.
    The question will be semantically related but NOT a copy of the text.
    """
    prompt_messages = [
        {
            "role": "system",
            "content": (
                "You are a legal exam question writer. "
                "Given a passage from a US Supreme Court opinion, "
                "write exactly ONE short factual question (max 20 words) "
                "that can be answered using only the passage. "
                "Output the question only. No explanation. No numbering."
            )
        },
        {
            "role": "user",
            "content": f"PASSAGE:\n{chunk_text[:600]}\n\nQUESTION:"
        }
    ]

    prompt = tok.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tok(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(gen.device)

    with torch.no_grad():
        output = gen.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=False,
            use_cache=True
        )

    generated_ids = output[0][inputs["input_ids"].shape[1]:]
    question = tok.decode(generated_ids, skip_special_tokens=True).strip()

    # Clean up — take only first line in case model adds explanation
    question = question.split("\n")[0].strip()
    return question


# ── Step 2: Sample chunks and generate questions

NUM_EVAL_SAMPLES = 100   # 100 is solid for a graduation project
random.seed(42)

sampled_indices = random.sample(range(len(docs)), NUM_EVAL_SAMPLES)

print(f"🔄 Generating {NUM_EVAL_SAMPLES} questions from sampled chunks...")
print("   This uses Qwen2.5-7B \n")

qa_pairs = []   # list of {chunk_idx, case_id, question}

for idx in tqdm(sampled_indices, desc="Generating questions"):
    chunk    = docs[idx]["chunk"]
    case_id  = docs[idx]["meta"]["case_id"]
    question = generate_question_from_chunk(chunk)

    qa_pairs.append({
        "chunk_idx" : idx,
        "case_id"   : case_id,
        "question"  : question,
        "chunk_text": chunk
    })

print(f"\n✅ Generated {len(qa_pairs)} Q&A pairs.")
print("\n📋 Sample questions:")
for pair in qa_pairs[:5]:
    print(f"   case_id={pair['case_id']} → {pair['question']}")



🔄 Generating 100 questions from sampled chunks...
   This uses Qwen2.5-7B 



Generating questions: 100%|██████████| 100/100 [01:49<00:00,  1.10s/it]


✅ Generated 100 Q&A pairs.

📋 Sample questions:
   case_id=1183 → What are the two general methods of taxation authorized by the Constitution of the State of Michigan according to the passage?
   case_id=220 → Did the passage suggest the complaint only involved interstate trade and commerce?
   case_id=2718 → What percentage of shareholders approved the merger in 1961?
   case_id=2418 → What year did the Secretary of Defense designate the shipyard as a 'defense facility'?
   case_id=2214 → Does the passage suggest a state violation of the Equal Protection Clause if voting is made dependent on affluence or fees?


# CELL 19 — 📊 Layer 1 Results: Proper Recall@K & MRR@K


In [ ]:
def evaluate_retrieval_proper(qa_pairs: list, k_list: tuple = (1, 3, 5, 10)) -> dict:
    results = {f"Recall@{k}": [] for k in k_list}
    results.update({f"MRR@{k}": [] for k in k_list})

    for pair in tqdm(qa_pairs, desc="Evaluating retrieval (proper)"):
        question    = pair["question"]
        target_case = pair["case_id"]

        # Full hybrid retrieval pipeline
        dense     = dense_retrieve(question, TOPK_DENSE)
        sparse    = bm25_retrieve(question, TOPK_BM25)
        fused_res = fuse(dense, sparse, ALPHA_DENSE, ALPHA_BM25, TOPK_FUSED)
        rr        = rerank(question, fused_res, TOPK_RERANK)
        ranked    = [idx for idx, _ in rr]

        for k in k_list:
            # Recall@K
            hit = any(docs[i]["meta"]["case_id"] == target_case for i in ranked[:k])
            results[f"Recall@{k}"].append(1.0 if hit else 0.0)

            # MRR@K
            mrr = 0.0
            for rank, chunk_idx in enumerate(ranked[:k], start=1):
                if docs[chunk_idx]["meta"]["case_id"] == target_case:
                    mrr = 1.0 / rank
                    break
            results[f"MRR@{k}"].append(mrr)

    return {metric: round(float(np.mean(vals)), 4) for metric, vals in results.items()}


print("🔄 Running proper retrieval evaluation...")
proper_retrieval_metrics = evaluate_retrieval_proper(qa_pairs, k_list=(1, 3, 5, 10))

print("\n📊 Proper Retrieval Evaluation Results (LLM-Generated Questions):")
print("─" * 50)
for metric, val in proper_retrieval_metrics.items():
    bar   = "█" * int(val * 20)
    empty = "░" * (20 - int(val * 20))
    print(f"   {metric:<12} {val:.4f}  [{bar}{empty}]")

# Compare with broken Cell 15 results for transparency
print("\n⚠️  Comparison with Cell 15 (data-leaked) results:")
print("   Cell 15 (leaked)  → all metrics = 1.0000  ← not meaningful")
print("   Cell 19 (proper)  → see above             ← real performance")


🔄 Running proper retrieval evaluation...


Evaluating retrieval (proper): 100%|██████████| 100/100 [04:19<00:00,  2.59s/it]


📊 Proper Retrieval Evaluation Results (LLM-Generated Questions):
──────────────────────────────────────────────────
   Recall@1     0.5800  [███████████░░░░░░░░░]
   Recall@3     0.7900  [███████████████░░░░░]
   Recall@5     0.8200  [████████████████░░░░]
   Recall@10    0.8500  [█████████████████░░░]
   MRR@1        0.5800  [███████████░░░░░░░░░]
   MRR@3        0.6667  [█████████████░░░░░░░]
   MRR@5        0.6742  [█████████████░░░░░░░]
   MRR@10       0.6785  [█████████████░░░░░░░]

⚠️  Comparison with Cell 15 (data-leaked) results:
   Cell 15 (leaked)  → all metrics = 1.0000  ← not meaningful
   Cell 19 (proper)  → see above             ← real performance


# CELL 20 — 🏗️ Layer 2: Build RAGAS Dataset


In [ ]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from datasets import Dataset

print("🔄 Building RAGAS evaluation dataset from your labeled queries...")
print(f"   Using {len(EVAL_QUERIES)} queries from Cell 16.\n")

ragas_data = {
    "question"    : [],
    "answer"      : [],
    "contexts"    : [],
    "ground_truth": [],
}

for q in tqdm(EVAL_QUERIES, desc="Running RAG pipeline for RAGAS"):
    # Full pipeline
    dense     = dense_retrieve(q, TOPK_DENSE)
    sparse    = bm25_retrieve(q, TOPK_BM25)
    fused_res = fuse(dense, sparse, ALPHA_DENSE, ALPHA_BM25, TOPK_FUSED)
    rr        = rerank(q, fused_res, TOPK_RERANK)

    # Skip if below threshold (same gate as answer())
    if not rr or rr[0][1] < RERANK_THRESHOLD:
        continue

    # Get retrieved contexts (full chunks, not clipped)
    contexts = [docs[idx]["chunk"] for idx, _ in rr]

    # Get generated answer
    resp, _ = answer(q)

    # Ground truth = source chunk of top-ranked result (best proxy without gold answers)
    ground_truth = docs[rr[0][0]]["chunk"]

    ragas_data["question"].append(q)
    ragas_data["answer"].append(resp)
    ragas_data["contexts"].append(contexts)
    ragas_data["ground_truth"].append(ground_truth)

ragas_dataset = Dataset.from_dict(ragas_data)

print(f"\n✅ RAGAS dataset built: {len(ragas_dataset)} samples")
print(f"   Skipped: {len(EVAL_QUERIES) - len(ragas_dataset)} queries (below score threshold)")



/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_804/1669111218.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/tmp/ipykernel_804/1669111218.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (

🔄 Building RAGAS evaluation dataset from your labeled queries...
   Using 20 queries from Cell 16.



Running RAG pipeline for RAGAS: 100%|██████████| 20/20 [02:37<00:00,  7.87s/it]


✅ RAGAS dataset built: 18 samples
   Skipped: 2 queries (below score threshold)


# CELL 21 — 🎯 Layer 2 Results: RAGAS Scores


In [ ]:
# Manual RAGAS-Style Metrics

def score_faithfulness(answer: str, contexts: list) -> float:
    """
    Ask Qwen: is every claim in this answer supported by these contexts?
    Returns score between 0.0 and 1.0.
    """
    context_str = "\n\n".join(contexts[:3])  # top 3 contexts
    messages = [
        {"role": "system", "content": (
            "You are an evaluation judge. "
            "Given an answer and source contexts, count how many claims "
            "in the answer are supported by the contexts. "
            "Reply with ONLY a number between 0.0 and 1.0. Nothing else."
        )},
        {"role": "user", "content": (
            f"CONTEXTS:\n{context_str[:1500]}\n\n"
            f"ANSWER:\n{answer}\n\n"
            f"Faithfulness score (0.0-1.0):"
        )}
    ]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=2048).to(gen.device)
    with torch.no_grad():
        out = gen.generate(**inputs, max_new_tokens=10, do_sample=False)
    generated = out[0][inputs["input_ids"].shape[1]:]
    text = tok.decode(generated, skip_special_tokens=True).strip()
    try:
        return min(1.0, max(0.0, float(text.split()[0])))
    except:
        return 0.5  # neutral fallback

def score_answer_relevancy(question: str, answer: str) -> float:
    """
    Embedding cosine similarity between question and answer.
    High similarity = answer is on-topic.
    """
    q_emb = embedder.encode([question], normalize_embeddings=True)
    a_emb = embedder.encode([answer],   normalize_embeddings=True)
    return float(np.dot(q_emb[0], a_emb[0]))

def score_context_precision(question: str, contexts: list) -> float:
    """
    Ask Qwen: is each retrieved context relevant to the question?
    Returns fraction of contexts judged relevant.
    """
    relevant = 0
    for ctx in contexts[:3]:
        messages = [
            {"role": "system", "content": (
                "You are an evaluation judge. "
                "Is this context relevant to answering the question? "
                "Reply with ONLY: YES or NO."
            )},
            {"role": "user", "content": f"QUESTION:\n{question}\n\nCONTEXT:\n{ctx[:800]}"}
        ]
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=1500).to(gen.device)
        with torch.no_grad():
            out = gen.generate(**inputs, max_new_tokens=5, do_sample=False)
        generated = out[0][inputs["input_ids"].shape[1]:]
        text = tok.decode(generated, skip_special_tokens=True).strip().upper()
        if "YES" in text:
            relevant += 1
    return relevant / min(len(contexts), 3)

def score_context_recall(answer: str, contexts: list) -> float:
    """
    Ask Qwen: does the answer use information from the contexts?
    Proxy for recall — did retrieval provide what was needed?
    """
    context_str = "\n\n".join(contexts[:3])
    messages = [
        {"role": "system", "content": (
            "You are an evaluation judge. "
            "Does this answer appear to be grounded in the provided contexts? "
            "Reply with ONLY a number between 0.0 and 1.0. Nothing else."
        )},
        {"role": "user", "content": (
            f"CONTEXTS:\n{context_str[:1500]}\n\n"
            f"ANSWER:\n{answer}\n\n"
            f"Context recall score (0.0-1.0):"
        )}
    ]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=2048).to(gen.device)
    with torch.no_grad():
        out = gen.generate(**inputs, max_new_tokens=10, do_sample=False)
    generated = out[0][inputs["input_ids"].shape[1]:]
    text = tok.decode(generated, skip_special_tokens=True).strip()
    try:
        return min(1.0, max(0.0, float(text.split()[0])))
    except:
        return 0.5

# ── Run evaluation ────────────────────────────────────────────────────────────
print("🔄 Running manual RAGAS-style evaluation...")
print(f"   Evaluating {len(ragas_dataset)} samples — expect ~15–20 min.\n")

faith_scores, relevancy_scores, precision_scores, recall_scores = [], [], [], []

for i, sample in enumerate(tqdm(ragas_dataset, desc="Scoring")):
    question = sample["question"]
    answer   = sample["answer"]
    contexts = sample["contexts"]

    faith_scores.append(score_faithfulness(answer, contexts))
    relevancy_scores.append(score_answer_relevancy(question, answer))
    precision_scores.append(score_context_precision(question, contexts))
    recall_scores.append(score_context_recall(answer, contexts))

ragas_scores = {
    "Faithfulness"     : round(float(np.mean(faith_scores)),     4),
    "Answer Relevancy" : round(float(np.mean(relevancy_scores)), 4),
    "Context Precision": round(float(np.mean(precision_scores)), 4),
    "Context Recall"   : round(float(np.mean(recall_scores)),    4),
}

print("\n📊 Manual RAGAS-Style Evaluation Results:")
print("─" * 50)
for metric, val in ragas_scores.items():
    bar   = "█" * int(val * 20)
    empty = "░" * (20 - int(val * 20))
    print(f"   {metric:<22} {val:.4f}  [{bar}{empty}]")

🔄 Running manual RAGAS-style evaluation...
   Evaluating 18 samples — expect ~15–20 min.



Scoring:   0%|          | 0/18 [00:00<?, ?it/s]Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Scoring:   6%|▌         | 1/18 [00:41<11:47, 41.60s/it]Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for 


📊 Manual RAGAS-Style Evaluation Results:
──────────────────────────────────────────────────
   Faithfulness           0.5278  [██████████░░░░░░░░░░]
   Answer Relevancy       0.6873  [█████████████░░░░░░░]
   Context Precision      0.7222  [██████████████░░░░░░]
   Context Recall         0.6833  [█████████████░░░░░░░]


# CELL 22 — 🏷️ Layer 3: Concept-Relevance@5 (from your Cell 16 labels)


In [ ]:

total_labeled  = 0
total_relevant = 0
concept_rel_at5 = None

if labels:
    rel_scores      = [l["relevant_at5"] for l in labels]
    concept_rel_at5 = round(float(np.mean(rel_scores)), 4)
    total_labeled   = len(rel_scores)
    total_relevant  = sum(rel_scores)

    print("📊 Manual Concept-Relevance@5 (from your Cell 16 labels):")
    print("─" * 50)
    print(f"   Concept-Relevance@5 : {concept_rel_at5:.4f}")
    print(f"   Relevant queries    : {total_relevant}/{total_labeled}")
else:
    print("⚠️  No labels found — run Cell 16 first to generate manual labels.")





📊 Manual Concept-Relevance@5 (from your Cell 16 labels):
──────────────────────────────────────────────────
   Concept-Relevance@5 : 0.9000
   Relevant queries    : 18/20


# CELL 23 — 📋 Full Evaluation Report (All Metrics Combined)


In [ ]:
# One clean printout combining all 3 evaluation layers.
# Copy this into your thesis / presentation. 🎓

def score_to_grade(val: float) -> str:
    if val >= 0.90: return "🟢 Excellent"
    if val >= 0.75: return "🟡 Good"
    if val >= 0.60: return "🟠 Acceptable"
    return "🔴 Needs Improvement"

print("=" * 65)
print("       ⚖️  LEGAL RAG SYSTEM — FULL EVALUATION REPORT")
print("=" * 65)

print("\n📌 LAYER 1 — Retrieval Quality (LLM-Generated Questions, n=100)")
print("─" * 65)
for metric, val in proper_retrieval_metrics.items():
    print(f"   {metric:<12} {val:.4f}   {score_to_grade(val)}")

print("\n📌 LAYER 2 — End-to-End Pipeline Quality (RAGAS, n={})".format(len(ragas_dataset)))
print("─" * 65)
for metric, val in ragas_scores.items():
    print(f"   {metric:<22} {val:.4f}   {score_to_grade(val)}")

print("\n📌 LAYER 3 — Human Judgment (Manual Labels, n={})".format(total_labeled if labels else 0))
print("─" * 65)
if concept_rel_at5 is not None:
    print(f"   {'Concept-Relevance@5':<22} {concept_rel_at5:.4f}   {score_to_grade(concept_rel_at5)}")
else:
    print("   No labels available.")

# Overall system score (weighted average across key metrics)
key_metrics = [
    proper_retrieval_metrics.get("Recall@5", 0),
    proper_retrieval_metrics.get("MRR@5", 0),
    ragas_scores.get("Faithfulness", 0),
    ragas_scores.get("Answer Relevancy", 0),
    ragas_scores.get("Context Precision", 0),
]
if concept_rel_at5 is not None:
    key_metrics.append(concept_rel_at5)

overall = round(float(np.mean(key_metrics)), 4)

print("\n" + "=" * 65)
print(f"   🏆 OVERALL SYSTEM SCORE   {overall:.4f}   {score_to_grade(overall)}")
print("=" * 65)

print("\n📎 Notes:")
print("   • Retrieval eval uses LLM-generated questions (no data leakage)")
print("   • RAGAS uses LLM-as-judge for automated scoring")
print("   • Manual labels provide human-in-the-loop validation")
print("   • Overall = unweighted mean of Recall@5, MRR@5,")
print("     Faithfulness, Answer Relevancy, Context Precision")
if concept_rel_at5 is not None:
    print("     + Concept-Relevance@5")



       ⚖️  LEGAL RAG SYSTEM — FULL EVALUATION REPORT

📌 LAYER 1 — Retrieval Quality (LLM-Generated Questions, n=100)
─────────────────────────────────────────────────────────────────
   Recall@1     0.5800   🔴 Needs Improvement
   Recall@3     0.7900   🟡 Good
   Recall@5     0.8200   🟡 Good
   Recall@10    0.8400   🟡 Good
   MRR@1        0.5800   🔴 Needs Improvement
   MRR@3        0.6667   🟠 Acceptable
   MRR@5        0.6742   🟠 Acceptable
   MRR@10       0.6775   🟠 Acceptable

📌 LAYER 2 — End-to-End Pipeline Quality (RAGAS, n=18)
─────────────────────────────────────────────────────────────────
   Faithfulness           0.5278   🔴 Needs Improvement
   Answer Relevancy       0.6873   🟠 Acceptable
   Context Precision      0.7222   🟠 Acceptable
   Context Recall         0.6833   🟠 Acceptable

📌 LAYER 3 — Human Judgment (Manual Labels, n=20)
─────────────────────────────────────────────────────────────────
   Concept-Relevance@5    0.9000   🟢 Excellent

   🏆 OVERALL SYSTEM SCORE   0.721

# CELL 25 — 💾 Save Everything to Google Drive — RAG102


In [ ]:
from google.colab import drive

# ── Mount Drive ───────────────────────────────────────────────────────────────
drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/RAG102"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"📁 Save directory ready: {SAVE_DIR}\n")


# ── 1. FAISS Index ────────────────────────────────────────────────────────────
faiss.write_index(index, f"{SAVE_DIR}/faiss_index.bin")
print(f"✅ FAISS index saved        → faiss_index.bin         ({index.ntotal} vectors)")


# ── 2. Docs + Metadata ───────────────────────────────────────────────────────
with open(f"{SAVE_DIR}/docs_meta.json", "w") as f:
    json.dump([d["meta"] for d in docs], f, indent=2)
print(f"✅ Docs metadata saved      → docs_meta.json          ({len(docs)} chunks)")


# ── 3. Full Docs with Chunks (for reloading without re-chunking) ──────────────
with open(f"{SAVE_DIR}/docs_full.json", "w") as f:
    json.dump(docs, f, indent=2)
print(f"✅ Full docs saved          → docs_full.json          ({len(docs)} chunks)")


# ── 4. Manual Labels (Cell 16) ────────────────────────────────────────────────
with open(f"{SAVE_DIR}/labels.json", "w") as f:
    json.dump(labels, f, indent=2)
print(f"✅ Manual labels saved      → labels.json             ({len(labels)} labeled queries)")


# ── 5. QA Pairs (Cell 18) ────────────────────────────────────────────────────
with open(f"{SAVE_DIR}/qa_pairs.json", "w") as f:
    json.dump(qa_pairs, f, indent=2)
print(f"✅ QA pairs saved           → qa_pairs.json           ({len(qa_pairs)} pairs)")


# ── 6. Proper Retrieval Metrics (Cell 19) ────────────────────────────────────
with open(f"{SAVE_DIR}/retrieval_metrics.json", "w") as f:
    json.dump(proper_retrieval_metrics, f, indent=2)
print(f"✅ Retrieval metrics saved  → retrieval_metrics.json")


# ── 7. RAGAS Scores (Cell 21) ────────────────────────────────────────────────
with open(f"{SAVE_DIR}/ragas_scores.json", "w") as f:
    json.dump(ragas_scores, f, indent=2)
print(f"✅ RAGAS scores saved       → ragas_scores.json")


# ── 8. Full Evaluation Report (all metrics combined) ─────────────────────────
full_report = {
    "system_info": {
        "dataset"        : "lex_glue/scotus",
        "max_cases"      : MAX_CASES,
        "total_chunks"   : len(docs),
        "embedder"       : "BAAI/bge-m3",
        "reranker"       : "BAAI/bge-reranker-large",
        "generator"      : GEN_MODEL,
        "chunk_chars"    : CHUNK_CHARS,
        "chunk_overlap"  : CHUNK_OVERLAP,
        "topk_dense"     : TOPK_DENSE,
        "topk_bm25"      : TOPK_BM25,
        "topk_fused"     : TOPK_FUSED,
        "topk_rerank"    : TOPK_RERANK,
        "alpha_dense"    : ALPHA_DENSE,
        "alpha_bm25"     : ALPHA_BM25,
    },
    "layer1_retrieval" : proper_retrieval_metrics,
    "layer2_ragas"     : ragas_scores,
    "layer3_human"     : {
        "concept_relevance_at5" : concept_rel_at5,
        "total_labeled"         : total_labeled,
        "total_relevant"        : total_relevant,
    },
    "overall_score"    : overall,
}

with open(f"{SAVE_DIR}/full_report.json", "w") as f:
    json.dump(full_report, f, indent=2)
print(f"✅ Full report saved        → full_report.json")


# ── 9. RAGAS Dataset (Cell 20) ───────────────────────────────────────────────
ragas_dataset.to_json(f"{SAVE_DIR}/ragas_dataset.jsonl")
print(f"✅ RAGAS dataset saved      → ragas_dataset.jsonl     ({len(ragas_dataset)} samples)")


# ── Final Summary ─────────────────────────────────────────────────────────────
print(f"""
{'='*55}
📁 RAG102 — Save Complete
{'='*55}
   faiss_index.bin       → FAISS vector index
   docs_meta.json        → chunk metadata only
   docs_full.json        → full chunks + metadata
   labels.json           → manual relevance labels
   qa_pairs.json         → LLM-generated Q&A pairs
   retrieval_metrics.json→ Recall@K, MRR@K results
   ragas_scores.json     → RAGAS 4 metrics
   full_report.json      → everything combined
   ragas_dataset.jsonl   → RAGAS evaluation dataset
{'='*55}
🏆 Overall Score: {overall:.4f}
📍 Location: {SAVE_DIR}
{'='*55}
""")

Mounted at /content/drive
📁 Save directory ready: /content/drive/MyDrive/RAG102

✅ FAISS index saved        → faiss_index.bin         (154628 vectors)
✅ Docs metadata saved      → docs_meta.json          (154628 chunks)
✅ Full docs saved          → docs_full.json          (154628 chunks)
✅ Manual labels saved      → labels.json             (20 labeled queries)
✅ QA pairs saved           → qa_pairs.json           (100 pairs)
✅ Retrieval metrics saved  → retrieval_metrics.json
✅ RAGAS scores saved       → ragas_scores.json
✅ Full report saved        → full_report.json


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

✅ RAGAS dataset saved      → ragas_dataset.jsonl     (18 samples)

📁 RAG102 — Save Complete
   faiss_index.bin       → FAISS vector index
   docs_meta.json        → chunk metadata only
   docs_full.json        → full chunks + metadata
   labels.json           → manual relevance labels
   qa_pairs.json         → LLM-generated Q&A pairs
   retrieval_metrics.json→ Recall@K, MRR@K results
   ragas_scores.json     → RAGAS 4 metrics
   full_report.json      → everything combined
   ragas_dataset.jsonl   → RAGAS evaluation dataset
🏆 Overall Score: 0.7219
📍 Location: /content/drive/MyDrive/RAG102



In [ ]:
# CELL — Restore answer() function
import torch

def answer(query: str) -> tuple:
    if is_smalltalk(query):
        return (
            "Hey! 👋 I'm your SCOTUS Legal Assistant. "
            "Ask me about constitutional law, landmark cases, or any US Supreme Court topic.",
            []
        )

    dense  = dense_retrieve(query, TOPK_DENSE)
    sparse = bm25_retrieve(query, TOPK_BM25)
    fused  = fuse(dense, sparse, ALPHA_DENSE, ALPHA_BM25, TOPK_FUSED)
    rr     = rerank(query, fused, TOPK_RERANK)

    if not rr or rr[0][1] < RERANK_THRESHOLD:
        return "Not found in the provided corpus.", rr

    sources      = format_sources(rr)
    user_message = f"QUESTION:\n{query}\n\nSOURCES:\n{sources}\n\nWrite the ANSWER only. End with <END>."
    messages     = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_message}
    ]

    prompt = tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tok(
        prompt, return_tensors="pt",
        truncation=True, max_length=MAX_INPUT_TOKENS
    ).to(gen.device)

    with torch.no_grad():
        output = gen.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True
        )

    generated_ids = output[0][inputs["input_ids"].shape[1]:]
    text          = tok.decode(generated_ids, skip_special_tokens=True)

    if "<END>" in text:
        text = text.split("<END>", 1)[0]

    return text.strip(), rr

print("✅ answer() restored successfully.")

✅ answer() restored successfully.


In [ ]:
# VERIFY CELL
resp, rr = answer("What is the Miranda rule for custodial interrogation?")
print("✅ answer() working!")
print("Response:", resp[:200])
print("Top score:", rr[0][1] if rr else "No results")

✅ answer() working!
Response: Not found in the provided corpus.
Top score: 0.6444061994552612


# ⚖️ LEGAL RAG SYSTEM UI
## Professional Gradio Interface

In [ ]:
!pip -q install gradio

import gradio as gr
import time
import torch

print("✅ UI dependencies ready.")


✅ UI dependencies ready.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 26 — 🎨 Simple UI v1
# ─────────────────────────────────────────────────────────────────────────────

import gradio as gr
import time

def simple_chat(query):
    if not query.strip():
        return "Please enter a question.", "—", "No sources."

    start    = time.time()
    resp, rr = answer(query)
    elapsed  = time.time() - start

    # Sources
    sources = ""
    for k, (idx, score) in enumerate(rr[:3], 1):
        meta    = docs[idx]["meta"]
        snippet = docs[idx]["chunk"][:300]
        sources += f"**[S{k}]** score={score:.3f} | case_id={meta['case_id']}\n{snippet}\n\n---\n\n"

    meta_text = f"**Time:** {elapsed:.2f}s  |  **Top Score:** {rr[0][1]:.4f}" if rr else "No results"

    return resp, meta_text, sources or "No sources retrieved."


demo = gr.Interface(
    fn=simple_chat,
    inputs=gr.Textbox(
        label="Ask a SCOTUS Legal Question",
        lines=2,
        placeholder="e.g. What is probable cause?"
    ),
    outputs=[
        gr.Textbox(label="Answer",   lines=6),
        gr.Textbox(label="Metadata", lines=1),
        gr.Markdown(label="Retrieved Sources"),
    ],
    title="⚖️ AI Legal RAG Assistant",
    description=(
        "Hybrid Retrieval (BM25 + BGE-M3) → "
        "Cross-Encoder Reranking → "
        "Grounded Generation (Qwen2.5-7B-Instruct)"
    ),
    examples=[
        ["What is the Miranda rule for custodial interrogation?"],
        ["Explain probable cause for a warrantless search."],
        ["What is qualified immunity for police officers?"],
        ["How does the exclusionary rule apply?"],
        ["What is the Strickland test?"],
    ],
)

demo.launch(share=True, show_error=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://aa9218f555a7717ed5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# ⚖️ FASTAPI BACKEND


# CELL 29 — 📦 Install Dependencies


In [ ]:
!pip -q install fastapi uvicorn pyngrok nest-asyncio

print("✅ FastAPI dependencies installed.")



✅ FastAPI dependencies installed.


# CELL 30 — ⚙️ Imports & Setup


In [ ]:

import time
import threading
import uvicorn
import nest_asyncio
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok
from pydantic import BaseModel, Field
from typing import Optional

# nest_asyncio allows uvicorn to run inside Colab's existing event loop
nest_asyncio.apply()

print("✅ Imports ready.")


✅ Imports ready.


# CELL 31 — 📐 Request & Response Models (Pydantic)


In [ ]:
# Pydantic automatically validates every incoming request.
# If the user sends wrong types, FastAPI returns a clear error — no crash.

class AskRequest(BaseModel):
    query : str            = Field(...,  min_length=1, max_length=1000,
                                   description="Legal question to ask")
    top_k : Optional[int]  = Field(3,   ge=1, le=6,
                                   description="Number of sources to return (1-6)")

class SourceRequest(BaseModel):
    query : str            = Field(...,  min_length=1, max_length=1000,
                                   description="Query to retrieve sources for")
    top_k : Optional[int]  = Field(5,   ge=1, le=10,
                                   description="Number of sources to return")

class ClassifyRequest(BaseModel):
    query : str            = Field(...,  min_length=1, max_length=1000,
                                   description="Query to classify")

class SourceItem(BaseModel):
    rank     : int
    case_id  : int
    chunk_id : int
    score    : float
    snippet  : str

class AskResponse(BaseModel):
    query      : str
    answer     : str
    intent     : str
    sources    : list[SourceItem]
    top_score  : float
    confidence : str
    time_ms    : int

class SourceResponse(BaseModel):
    query   : str
    sources : list[SourceItem]
    time_ms : int

class ClassifyResponse(BaseModel):
    query   : str
    intent  : str
    time_ms : int

class HealthResponse(BaseModel):
    status      : str
    model       : str
    total_chunks: int
    total_cases : int
    embedder    : str
    reranker    : str

print("✅ Pydantic models ready.")


✅ Pydantic models ready.


# CELL 32 — 🔧 Helper Functions


In [ ]:

def get_confidence(score: float) -> str:
    if score >= 0.92: return "Very High"
    if score >= 0.85: return "High"
    if score >= 0.75: return "Medium"
    if score >= 0.60: return "Low"
    return "Very Low"

def build_source_items(rr: list, top_k: int) -> list:
    """Convert reranked results into SourceItem list."""
    items = []
    for rank, (idx, score) in enumerate(rr[:top_k], 1):
        meta = docs[idx]["meta"]
        items.append(SourceItem(
            rank     = rank,
            case_id  = meta["case_id"],
            chunk_id = meta["chunk_id"],
            score    = round(float(score), 4),
            snippet  = docs[idx]["chunk"][:400]
        ))
    return items

def run_retrieval(query: str) -> list:
    """Full hybrid retrieval pipeline — dense + BM25 + fusion + rerank."""
    dense     = dense_retrieve(query, TOPK_DENSE)
    sparse    = bm25_retrieve(query, TOPK_BM25)
    fused_res = fuse(dense, sparse, ALPHA_DENSE, ALPHA_BM25, TOPK_FUSED)
    return rerank(query, fused_res, TOPK_RERANK)

print("✅ Helper functions ready.")

✅ Helper functions ready.


# CELL 33 — 🚀 FastAPI App & Endpoints


In [ ]:

app = FastAPI(
    title       = "⚖️ Legal RAG API",
    description = (
        "Hybrid RAG system for US Supreme Court legal questions.\n\n"
        "**Pipeline:** BM25 + BGE-M3 → Fusion → BGE-reranker-large → Qwen2.5-7B\n\n"
        "**Dataset:** lex_glue/scotus — 5000 cases"
    ),
    version     = "1.0.0",
)

# ── CORS — allow any website to call this API ─────────────────────────────────
app.add_middleware(
    CORSMiddleware,
    allow_origins     = ["*"],   # in production: restrict to your domain
    allow_credentials = True,
    allow_methods     = ["*"],
    allow_headers     = ["*"],
)
# ─────────────────────────────────────────────────────────────────────────────
# ENDPOINT 1 — GET /health
# Check if the system is running and models are loaded.
# ─────────────────────────────────────────────────────────────────────────────

@app.get("/health", response_model=HealthResponse, tags=["System"])
def health():
    """
    Health check — confirms all models are loaded and system is ready.
    Call this first to verify the API is alive.
    """
    return HealthResponse(
        status       = "healthy",
        model        = GEN_MODEL,
        total_chunks = len(docs),
        total_cases  = len(cases),
        embedder     = "BAAI/bge-m3",
        reranker     = "BAAI/bge-reranker-large",
    )


# ─────────────────────────────────────────────────────────────────────────────
# ENDPOINT 2 — POST /ask
# Main endpoint — full RAG pipeline, returns answer + sources + metadata.
# ─────────────────────────────────────────────────────────────────────────────

@app.post("/ask", response_model=AskResponse, tags=["RAG"])
def ask(req: AskRequest):
    """
    Full RAG pipeline endpoint.

    - Classifies query as LEGAL or GENERAL
    - Runs hybrid retrieval + reranking
    - Generates grounded answer using Qwen2.5-7B
    - Returns answer, sources, confidence, and timing
    """
    start = time.time()

    try:
        # Smalltalk check
        if is_smalltalk(req.query):
            return AskResponse(
                query      = req.query,
                answer     = "Hey! I'm your SCOTUS Legal Assistant. Ask me about US Supreme Court law.",
                intent     = "SMALLTALK",
                sources    = [],
                top_score  = 0.0,
                confidence = "N/A",
                time_ms    = int((time.time() - start) * 1000)
            )

        # Classify
        intent = classify_query(req.query)

        # General query
        if intent == "GENERAL":
            response = handle_general_query(req.query)
            return AskResponse(
                query      = req.query,
                answer     = response,
                intent     = "GENERAL",
                sources    = [],
                top_score  = 0.0,
                confidence = "N/A",
                time_ms    = int((time.time() - start) * 1000)
            )

        # Full RAG pipeline
        resp, rr = answer(req.query)
        top_score = rr[0][1] if rr else 0.0

        return AskResponse(
            query      = req.query,
            answer     = resp,
            intent     = "LEGAL",
            sources    = build_source_items(rr, req.top_k),
            top_score  = round(top_score, 4),
            confidence = get_confidence(top_score),
            time_ms    = int((time.time() - start) * 1000)
        )

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


# ─────────────────────────────────────────────────────────────────────────────
# ENDPOINT 3 — POST /sources
# Retrieval only — no generation. Fast, cheap, useful for previewing sources.
# ─────────────────────────────────────────────────────────────────────────────

@app.post("/sources", response_model=SourceResponse, tags=["RAG"])
def sources(req: SourceRequest):
    """
    Retrieval-only endpoint — no generation.
    Returns top-K retrieved and reranked sources for a query.
    Much faster than /ask since it skips the LLM generation step.
    """
    start = time.time()

    try:
        rr = run_retrieval(req.query)
        return SourceResponse(
            query   = req.query,
            sources = build_source_items(rr, req.top_k),
            time_ms = int((time.time() - start) * 1000)
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


# ─────────────────────────────────────────────────────────────────────────────
# ENDPOINT 4 — POST /classify
# Intent classification only — LEGAL, GENERAL, or SMALLTALK.
# ─────────────────────────────────────────────────────────────────────────────

@app.post("/classify", response_model=ClassifyResponse, tags=["Utility"])
def classify(req: ClassifyRequest):
    """
    Query intent classification endpoint.
    Returns LEGAL, GENERAL, or SMALLTALK.
    Useful for routing on the frontend before deciding what to display.
    """
    start = time.time()

    try:
        if is_smalltalk(req.query):
            intent = "SMALLTALK"
        else:
            intent = classify_query(req.query)

        return ClassifyResponse(
            query   = req.query,
            intent  = intent,
            time_ms = int((time.time() - start) * 1000)
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


print("✅ FastAPI app and all endpoints ready.")
print("\n📋 Endpoints:")
print("   GET  /health    → system status")
print("   POST /ask       → full RAG pipeline")
print("   POST /sources   → retrieval only")
print("   POST /classify  → intent classification")
print("   GET  /docs      → interactive API docs (auto-generated)")




✅ FastAPI app and all endpoints ready.

📋 Endpoints:
   GET  /health    → system status
   POST /ask       → full RAG pipeline
   POST /sources   → retrieval only
   POST /classify  → intent classification
   GET  /docs      → interactive API docs (auto-generated)


# CELL 34 — 🌐 Launch Server + ngrok Tunnel


In [ ]:
# uvicorn runs FastAPI in a background thread so Colab stays responsive.
# ngrok creates a public HTTPS URL pointing to your local server.
#
# ⚠️  Get your free ngrok token at: https://dashboard.ngrok.com/get-started/your-authtoken
# ⚠️  Paste it below where it says YOUR_NGROK_TOKEN

NGROK_TOKEN = "33O5fDyI3VK6t3KT7Uvfiu6gpFa_2awnFyvme9tTAQDFxLveh"   # ← paste your token here
PORT        = 8000

# ── Authenticate ngrok ────────────────────────────────────────────────────────
ngrok.set_auth_token(NGROK_TOKEN)

# ── Start uvicorn in background thread ───────────────────────────────────────
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="error")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(3)   # wait for server to start

# ── Open ngrok tunnel ─────────────────────────────────────────────────────────
tunnel     = ngrok.connect(PORT)
public_url = tunnel.public_url

print("=" * 60)
print("🚀 Legal RAG API is LIVE!")
print("=" * 60)
print(f"\n🌐 Public URL : {public_url}")
print(f"\n📋 Endpoints  :")
print(f"   GET  {public_url}/health")
print(f"   POST {public_url}/ask")
print(f"   POST {public_url}/sources")
print(f"   POST {public_url}/classify")
print(f"\n📖 Interactive Docs (try it in browser):")
print(f"   {public_url}/docs")
print("\n⚠️  URL is valid while this Colab session is running.")
print("=" * 60)



ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use


🚀 Legal RAG API is LIVE!

🌐 Public URL : https://advised-felica-stubbornly.ngrok-free.dev

📋 Endpoints  :
   GET  https://advised-felica-stubbornly.ngrok-free.dev/health
   POST https://advised-felica-stubbornly.ngrok-free.dev/ask
   POST https://advised-felica-stubbornly.ngrok-free.dev/sources
   POST https://advised-felica-stubbornly.ngrok-free.dev/classify

📖 Interactive Docs (try it in browser):
   https://advised-felica-stubbornly.ngrok-free.dev/docs

⚠️  URL is valid while this Colab session is running.


# CELL 35 — 🧪 Test All Endpoints


In [ ]:
# Run this to confirm every endpoint works before connecting your website.

import requests

BASE = public_url
print(f"🧪 Testing API at: {BASE}\n")

# ── Test 1: Health ────────────────────────────────────────────────────────────
print("─" * 50)
print("1️⃣  GET /health")
r = requests.get(f"{BASE}/health")
print(f"   Status : {r.status_code}")
print(f"   Body   : {r.json()}")

# ── Test 2: Classify ──────────────────────────────────────────────────────────
print("\n─" * 50)
print("2️⃣  POST /classify")
r = requests.post(f"{BASE}/classify", json={"query": "What is probable cause?"})
print(f"   Status : {r.status_code}")
print(f"   Body   : {r.json()}")

# ── Test 3: Sources ───────────────────────────────────────────────────────────
print("\n─" * 50)
print("3️⃣  POST /sources")
r = requests.post(f"{BASE}/sources", json={"query": "miranda custodial interrogation", "top_k": 2})
data = r.json()
print(f"   Status  : {r.status_code}")
print(f"   Time    : {data['time_ms']}ms")
print(f"   Sources : {len(data['sources'])} returned")
for s in data["sources"]:
    print(f"     [S{s['rank']}] case_id={s['case_id']} score={s['score']}")

# ── Test 4: Ask ───────────────────────────────────────────────────────────────
print("\n─" * 50)
print("4️⃣  POST /ask  (this takes ~30s — runs full RAG pipeline)")
r = requests.post(
    f"{BASE}/ask",
    json={"query": "What is probable cause for a warrantless search?", "top_k": 3},
    timeout=120
)
data = r.json()
print(f"   Status     : {r.status_code}")
print(f"   Intent     : {data['intent']}")
print(f"   Confidence : {data['confidence']}")
print(f"   Top Score  : {data['top_score']}")
print(f"   Time       : {data['time_ms']}ms")
print(f"   Answer     : {data['answer'][:300]}")
print(f"   Sources    : {len(data['sources'])} returned")

print("\n✅ All endpoints tested successfully!")



🧪 Testing API at: https://advised-felica-stubbornly.ngrok-free.dev

──────────────────────────────────────────────────
1️⃣  GET /health
   Status : 200
   Body   : {'status': 'healthy', 'model': 'Qwen/Qwen2.5-7B-Instruct', 'total_chunks': 154628, 'total_cases': 4747, 'embedder': 'BAAI/bge-m3', 'reranker': 'BAAI/bge-reranker-large'}

─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
2️⃣  POST /classify
   Status : 200
   Body   : {'query': 'What is probable cause?', 'intent': 'LEGAL', 'time_ms': 246}

─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
3️⃣  POST /sources
   Status  : 200
   Time    : 1846ms
   Sources : 2 returned
     [S1] case_id=4336 score=0.948
     [S2] case_id=4336 score=0.9429

─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
4️⃣  POST /ask  (this takes ~30s — runs full RAG pipeline)
   Status     : 200
   Intent    